# Citewise GPU (Kaggle)

Runs the chat model for Citewise on Kaggle's free GPU. While this notebook runs, the Citewise server sends chat
here; when it stops, the server falls back to its small local model automatically.

**One-time setup**
1. *Settings* → **Accelerator: GPU T4 x2**, **Internet: on** (needs a phone-verified Kaggle account).
2. *Add-ons → Secrets* → add **`CITEWISE_GPU_TOKEN`** = the `GPU_TOKEN` from the server's `deploy/.env`, and tick it for this notebook.

**On demand:** if the server has a Kaggle API token (`deploy/kaggle-secrets/access_token`), it starts this notebook
when someone chats, and the notebook stops itself after `IDLE_STOP_MINUTES` without chat.

**To start it by hand** (uses your weekly GPU hours while it runs, up to ~12 h per run):
*Save Version* → **Save & Run All (Commit)**. It keeps running in the background; stop it from *Your Work → Versions*.

In [ ]:
MODEL = "qwen2.5:14b"   # best quality that still answers fast on one T4; "qwen2.5:32b" fits on T4 x2 but is slower
SERVER = "https://130-210-13-215.sslip.io"
RUN_HOURS = 11.5        # stop before Kaggle's session limit
IDLE_STOP_MINUTES = 5   # shut down (stop using GPU hours) after this long without chat

from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("CITEWISE_GPU_TOKEN")

In [ ]:
import os, subprocess, time, urllib.request
subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null", shell=True, check=True)
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version"); break
    except OSError:
        time.sleep(1)
subprocess.run(["ollama", "pull", MODEL], env=env, check=True)
subprocess.run(["ollama", "run", MODEL, "Say OK."], env=env, check=True)   # load it onto the GPU now

In [ ]:
# Token-checking proxy in front of Ollama, so only the Citewise server can use this GPU.
import hmac, http.client, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class Guard(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.0"
    def _relay(self):
        if not hmac.compare_digest(self.headers.get("Authorization", ""), f"Bearer {TOKEN}"):
            self.send_response(401); self.end_headers(); return
        body = self.rfile.read(int(self.headers.get("Content-Length") or 0)) or None
        conn = http.client.HTTPConnection("127.0.0.1", 11434, timeout=600)
        conn.request(self.command, self.path, body=body, headers={"Content-Type": "application/json"})
        resp = conn.getresponse()
        self.send_response(resp.status)
        self.send_header("Content-Type", resp.getheader("Content-Type", "application/json"))
        self.end_headers()
        while chunk := resp.read1(65536):
            self.wfile.write(chunk); self.wfile.flush()
    do_GET = do_POST = _relay
    def log_message(self, *args): pass

threading.Thread(target=ThreadingHTTPServer(("127.0.0.1", 8000), Guard).serve_forever, daemon=True).start()

In [ ]:
# Public HTTPS tunnel (Cloudflare quick tunnel, no account needed).
import re
subprocess.run("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0); break
print("tunnel:", URL)

In [ ]:
# Tell the Citewise server where this GPU is every minute. Stop when chat has been idle for
# IDLE_STOP_MINUTES (the server starts this notebook again on the next chat), or at RUN_HOURS.
import json
def register():
    req = urllib.request.Request(f"{SERVER}/gpu/register", method="POST",
                                 data=json.dumps({"url": URL, "model": MODEL}).encode(),
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)

ready = time.time()
end = ready + RUN_HOURS * 3600
while time.time() < end:
    try:
        reply = register()
        chat_idle = reply.get("idle_seconds")
        idle = time.time() - ready if chat_idle is None else min(chat_idle, time.time() - ready)
        print(time.strftime("%H:%M"), f"registered, idle {int(idle)}s", flush=True)
        if idle > IDLE_STOP_MINUTES * 60:
            print("No chat for", IDLE_STOP_MINUTES, "minutes; shutting down to save GPU hours.")
            break
    except Exception as e:
        print(time.strftime("%H:%M"), "register failed:", e, flush=True)
    time.sleep(60)
